# 📚 Research Paper Explainer

## What We're Building

A tool that takes a **research paper PDF** and generates:
1. **Plain-English Summary** — Explains the paper as if to a non-expert
2. **Glossary** — Defines technical terms used in the paper
3. **Quiz** — Multiple-choice questions to test understanding
4. **Interactive Q&A** — Ask any question about the paper

## What's Different from Project 01 (PDF Q&A)?

| PDF Q&A (Project 01) | Research Paper Explainer |
|---------------------|------------------------|
| General PDF Q&A | Specialized for academic papers |
| Simple retrieve-and-answer | Multi-stage pipeline (summarize, glossary, quiz) |
| LangChain only | LlamaIndex for indexing + LangChain for generation |
| Single prompt | Multiple specialized prompts |

## Stack
- **LangChain** — prompt chains and LLM orchestration
- **ChromaDB** — vector storage for RAG
- **Ollama** — local LLM and embeddings
- **PyPDF** — PDF text extraction

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install langchain langchain-ollama langchain-community chromadb pypdf -q

import urllib.request
from pathlib import Path

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
import json

print("All imports successful!")

## Step 2 — Load and Index the Paper

We'll use a well-known paper for testing: **"Attention Is All You Need"** (Transformer paper).

In [ ]:
# Download sample paper
pdf_url = "https://arxiv.org/pdf/1706.03762v7"
pdf_path = Path("research_paper.pdf")

if not pdf_path.exists():
    print("Downloading paper...")
    urllib.request.urlretrieve(pdf_url, pdf_path)

# Load and split
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()
print(f"Loaded {len(pages)} pages")

# Chunk with academic-friendly settings
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,    # Larger chunks for academic content
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " "],
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

# Build vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="research_paper",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print(f"Vector store ready: {vectorstore._collection.count()} vectors")

## Step 3 — Initialize the LLM

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.3)
print("LLM ready!")

## Step 4 — Generate Plain-English Summary

The first stage: take the full paper content and produce a summary that
a non-expert can understand. We use the **map-reduce** approach:
1. **Map**: Summarize each chunk independently
2. **Reduce**: Combine chunk summaries into one coherent summary

For simplicity, we'll retrieve the most important chunks and summarize those.

In [ ]:
# Get the most important chunks (abstract, introduction, conclusion)
key_queries = [
    "What is the main contribution and motivation of this paper?",
    "What is the proposed method or architecture?",
    "What are the experimental results and conclusions?",
]

# Retrieve relevant chunks for each query
key_chunks = []
seen_content = set()
for query in key_queries:
    docs = retriever.invoke(query)
    for doc in docs:
        # Deduplicate
        content_hash = hash(doc.page_content[:100])
        if content_hash not in seen_content:
            key_chunks.append(doc)
            seen_content.add(content_hash)

print(f"Collected {len(key_chunks)} unique key chunks")

# Combine key content
paper_context = "\n\n---\n\n".join(chunk.page_content for chunk in key_chunks)

# Summary prompt
summary_prompt = ChatPromptTemplate.from_template(
    """You are a science communicator who explains research papers to a general audience.

Read the following excerpts from a research paper and write a plain-English summary.

Your summary should include:
1. **What problem does this paper solve?** (1-2 sentences)
2. **What's the key idea?** (2-3 sentences, use analogies if helpful)
3. **How does it work?** (3-4 sentences, simplified)
4. **Why does it matter?** (1-2 sentences on impact)

Use simple language. Avoid jargon. If you must use a technical term, explain it inline.

Paper excerpts:
{context}

Plain-English summary:"""
)

summary_chain = summary_prompt | llm | StrOutputParser()

print("\n📝 Generating plain-English summary...\n")
summary = summary_chain.invoke({"context": paper_context})
print(summary)

## Step 5 — Generate a Glossary

Extract and define all technical terms from the paper.

In [ ]:
glossary_prompt = ChatPromptTemplate.from_template(
    """You are a technical glossary generator. From the following research paper excerpts,
identify and define all important technical terms.

For each term:
- Provide a simple, 1-2 sentence definition
- Use an analogy or example if it helps
- Note how it's used in this specific paper

Format as:
**Term**: Definition. _In this paper: how it's used._

Paper excerpts:
{context}

Glossary (list 10-15 key terms):"""
)

glossary_chain = glossary_prompt | llm | StrOutputParser()

print("📖 Generating glossary...\n")
glossary = glossary_chain.invoke({"context": paper_context})
print(glossary)

## Step 6 — Generate a Comprehension Quiz

Create multiple-choice questions to test understanding of the paper.

In [ ]:
class QuizQuestion(BaseModel):
    """A single quiz question."""
    question: str = Field(description="The question text")
    options: list[str] = Field(description="4 multiple choice options (A, B, C, D)")
    correct_answer: str = Field(description="The correct option letter (A, B, C, or D)")
    explanation: str = Field(description="Why the correct answer is right")


class Quiz(BaseModel):
    """A quiz about the research paper."""
    questions: list[QuizQuestion] = Field(description="5 quiz questions")


quiz_parser = JsonOutputParser(pydantic_object=Quiz)

quiz_prompt = ChatPromptTemplate.from_template(
    """You are a professor creating a quiz about a research paper.

Create 5 multiple-choice questions that test understanding of the key concepts.
Questions should range from basic comprehension to deeper understanding.

Mix of question types:
- 2 factual questions ("What does the paper propose?")
- 2 conceptual questions ("Why is X better than Y?")
- 1 application question ("In what scenario would you use...")

{format_instructions}

Paper excerpts:
{context}

Quiz as JSON:"""
)

quiz_chain = quiz_prompt | llm | quiz_parser

print("🧪 Generating quiz...\n")
quiz = quiz_chain.invoke({
    "context": paper_context,
    "format_instructions": quiz_parser.get_format_instructions(),
})

# Display the quiz
for i, q in enumerate(quiz.get("questions", []), 1):
    print(f"\nQ{i}: {q['question']}")
    for opt in q['options']:
        print(f"   {opt}")
    print(f"   Answer: {q['correct_answer']}")
    print(f"   💡 {q['explanation']}")

## Step 7 — Interactive Q&A

Ask any question about the paper — just like Project 01, but with
a prompt tuned for academic content.

In [ ]:
from langchain.prompts import PromptTemplate

academic_qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert research assistant. Answer questions about a research paper
using the provided context. 

Guidelines:
- Give accurate, detailed answers
- Use simple language where possible
- Cite specific sections ("According to Section 3...")
- If the context doesn't contain the answer, say so

Context from the paper:
{context}

Question: {question}

Answer:"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": academic_qa_prompt},
)


def ask_paper(question: str) -> None:
    """Ask a question about the research paper."""
    print(f"❓ {question}")
    result = qa_chain.invoke({"query": question})
    print(f"\n💡 {result['result']}")
    pages_cited = set(d.metadata.get('page', '?') for d in result['source_documents'])
    print(f"\n📄 Referenced pages: {sorted(pages_cited)}")
    print("=" * 60 + "\n")


ask_paper("What is self-attention and why is it important in this paper?")

In [ ]:
ask_paper("What are the advantages of the Transformer over RNNs and CNNs?")

In [ ]:
ask_paper("What training techniques and hyperparameters were used?")

## Step 8 — Complete Report Generator

Generate all outputs (summary, glossary, quiz) and save as a single report.

In [ ]:
report = f"""# Research Paper Explainer Report

## Plain-English Summary

{summary}

## Glossary of Key Terms

{glossary}

## Comprehension Quiz

"""

for i, q in enumerate(quiz.get("questions", []), 1):
    report += f"\n**Q{i}: {q['question']}**\n"
    for opt in q['options']:
        report += f"- {opt}\n"
    report += f"\n<details><summary>Answer</summary>{q['correct_answer']} — {q['explanation']}</details>\n"

# Save
Path("paper_explainer_report.md").write_text(report, encoding="utf-8")
print(f"📄 Report saved to paper_explainer_report.md")
print(f"   Length: {len(report)} characters")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Multi-query retrieval** | Uses multiple queries to gather diverse context |
| **Deduplication** | Removes duplicate chunks from multiple retrievals |
| **Specialized prompts** | Different prompts for summary, glossary, quiz |
| **Pydantic + JSON parser** | Structured quiz output with validation |
| **Multi-stage pipeline** | Chain multiple generation steps sequentially |

## 🔧 Customization Ideas

- **Difficulty levels**: Generate easy/medium/hard summaries
- **Comparison mode**: Compare two papers side-by-side
- **Citation graph**: Extract and visualize paper references
- **Figure extraction**: Use vision models to explain paper figures